## 1. Import our own implementation

In [ ]:
from pathlib import Path
from pprint import pprint

from resume_screening import (
    read_text_file,
    load_resume_folder,
    remove_personal_information_lines,
    tokenize,
    find_skills,
    extract_education,
    extract_experience_years,
    parse_job_requirements,
    build_tfidf_vectors,
    cosine_similarity,
    calculate_text_similarities,
    screen_resumes,
    print_results,
    export_results_csv,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "resume_screening.py").exists():
    raise FileNotFoundError(
        "Open this notebook with the project folder as the working directory."
    )

JOB_DESCRIPTION_PATH = PROJECT_ROOT / "data" / "job_description.txt"
RESUME_FOLDER = PROJECT_ROOT / "data" / "resumes"
OUTPUT_FOLDER = PROJECT_ROOT / "outputs"
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Python standard-library implementation loaded.")


## 2. Load the job description and resumes



In [ ]:
job_description = read_text_file(JOB_DESCRIPTION_PATH)
resumes = load_resume_folder(RESUME_FOLDER)

print("JOB DESCRIPTION")
print("-" * 60)
print(job_description)

print("\nRESUME FILES")
for candidate_id, resume_text in resumes.items():
    print(f"{candidate_id}: {len(resume_text)} characters")


## 3. Extract job requirements



In [ ]:
requirements = parse_job_requirements(job_description)
pprint(requirements)


## 4. Inspect candidate information


In [ ]:
for candidate_id, resume_text in resumes.items():
    print("\n", candidate_id)
    print("Skills:", sorted(find_skills(resume_text)))
    print("Education:")
    pprint(extract_education(resume_text))
    print("Explicit experience years:", extract_experience_years(resume_text))


## 5. Text preprocessing and tokenization

.

In [ ]:
example_resume = next(iter(resumes.values()))
print("Original excerpt:")
print(example_resume[:350])
print("\nTokens used for text similarity:")
print(tokenize(example_resume)[:60])


## 6. Build TF-IDF vectors ourselves



In [ ]:
documents = [job_description] + list(resumes.values())
tfidf_vectors = build_tfidf_vectors(documents)

print("Number of document vectors:", len(tfidf_vectors))
print("Number of distinct terms in the job-description vector:", len(tfidf_vectors[0]))
print("A few job-description TF-IDF values:")
for term, value in list(tfidf_vectors[0].items())[:15]:
    print(f"{term:20s} {value:.4f}")


## 7. Calculate cosine similarity



In [ ]:
job_vector = tfidf_vectors[0]
print("Text similarities (0 to 1):")
for (candidate_id, _), vector in zip(resumes.items(), tfidf_vectors[1:]):
    similarity = cosine_similarity(job_vector, vector)
    print(f"{candidate_id:20s} {similarity:.4f}")


## 8. Calculate scores and rank candidates



In [ ]:
report = screen_resumes(job_description, resumes)
print_results(report)


## 9. Inspect one candidate's explanation



In [ ]:
for candidate in report["candidates"]:
    print("\n" + "=" * 70)
    print(f"Rank {candidate['rank']}: {candidate['candidate_id']}")
    print("Overall score:", candidate["overall_score"], "%")
    print("Component scores:")
    pprint(candidate["components"])
    print("Matched required skills:", candidate["required_skills_matched"])
    print("Missing required skills:", candidate["required_skills_missing"])
    print("Matched preferred skills:", candidate["preferred_skills_matched"])
    print("Education:")
    pprint(candidate["education"])
    print("Experience evidence:", candidate["experience_evidence"])


## 10. Export a CSV report



In [ ]:
csv_path = export_results_csv(
    report, OUTPUT_FOLDER / "candidate_screening_results.csv"
)
print("CSV report written to:", csv_path)


## 11. Try your own input


In [ ]:

custom_job_description = """
Required Skills: Python, SQL, Microsoft Excel
Preferred Skills: Power BI, Pandas
At least 1 year of professional experience is required.
Bachelor's degree in Computer Science or Statistics preferred.
"""

custom_resumes = {
    "candidate_demo_001": """
    Skills: Python, SQL, Excel, Pandas
    Experience: 2 years of professional experience.
    Education: Bachelor of Science in Statistics.
    """,
    "candidate_demo_002": """
    Skills: Excel, Power BI
    Experience: 1 year of professional experience.
    Education: Diploma in Business Administration.
    """,
}

custom_report = screen_resumes(custom_job_description, custom_resumes)
print_results(custom_report)
